# Figures and tables of the paper

Every figure and table under `figures/` is produced here from the stored runs:
`paper.py` names the settings and paths, `stats.py` loads and computes,
`render.py` typesets. Runs from this directory; `ANALYSIS_OUT` redirects the
output root (default `figures/`).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, LogLocator, MaxNLocator, NullFormatter
import numpy as np
import pandas as pd
import seaborn as sns

import paper
import render
import stats

paper.style()
OUT = Path(os.environ.get("ANALYSIS_OUT", paper.FIGURES))
TABLES = OUT / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
SAVE_KW = paper.SAVE_KW

## Results

In [ ]:
results = paper.iter_results()
for name, emit in [
    ("coling_results", render.coling_results),
    ("prdb_indomain", render.prdb_indomain),
    ("prdb_full", render.prdb_full),
]:
  emit(results, "logreg", TABLES / f"{name}.tex")
results.pivot_table(
    index="model",
    columns=["dataset", "split"],
    values="threshold",
    aggfunc="nunique",
).fillna(0).astype(int)

In [ ]:
LINE_STYLES = {
    "neural": {"color": "black", "linestyle": ":", "label": "Neural Baseline"},
    "linguistic": {
        "color": "gray",
        "linestyle": "--",
        "label": "Linguistic Baseline",
    },
}
TITLE_TEXT = paper.COLING_SPLIT_LABEL | {
    "qwen": "Qwen 2.5 72B",
    "gemini": "Gemini 1.5 Pro",
    "gpt4o": "GPT-4o",
    "llama": "Llama 3.1 70B",
    "claude": "Claude Sonnet 3.5",
}
PALETTE_FIG1 = {
    "ft_model_sae": "tab:blue",
    "ft_sae": "#00A896",
    "base_sae": "#E8451A",
}
PALETTE_FIG2 = {
    "base_sae": "tab:blue",
    "ft_model_sae": "#00A896",
    "ft_sae": "#E8451A",
}
SQRT_SCALE = (
    lambda x: 1 - np.sqrt(np.clip(1 - x, 0, None)),
    lambda y: 1 - (1 - y) ** 2,
)
DATA_COLS = ["dataset", "split", "target", "model", "type", "head", "threshold"]
HEAD_FILTER = {
    "xgboost": ["xgboost", "baseline"],
    "logreg": ["logreg", "baseline"],
}
SUFFIX = {"xgboost": "", "logreg": "_logreg"}
RAID_STEM = {"xgboost": "raid_panel_xgb", "logreg": "raid_panel_logreg"}


def draw_panel(ax, d, metric, palette, title, title_size, legend_size, legend):
  ax.set_title(title, fontsize=title_size, fontweight="semibold")
  baselines = d[d["type"].isin(LINE_STYLES)]
  curves = d[~d["type"].isin(LINE_STYLES)]
  for _, row in baselines.iterrows():
    ax.axhline(y=row[metric], linewidth=2, **LINE_STYLES[row["type"]])
  sns.lineplot(
      data=curves,
      x="threshold",
      y=metric,
      marker="o",
      hue="type",
      ax=ax,
      palette=palette,
  )
  ax.set(xlabel="", ylabel="")
  if not legend:
    ax.get_legend().remove()
    return
  leg = ax.legend(
      loc="lower left",
      fontsize=legend_size,
      shadow=True,
      frameon=True,
      title="",
  )
  for t in leg.get_texts():
    t.set_text(paper.SAE_LABEL.get(t.get_text(), t.get_text()))


def coling_raid_frames(df):
  d = df[df["target"] == "all"]
  frames = [
      (s, d[(d["dataset"] == "COLING") & (d["split"] == s)])
      for s in ("dev", "devtest", "test")
  ]
  return frames + [("raid", d[d["dataset"] == "RAID"])]


def save(fig, data, stem):
  fig.savefig(OUT / f"{stem}.pdf", **SAVE_KW)
  plt.close(fig)
  data.sort_values(DATA_COLS)[DATA_COLS + paper.METRICS].to_csv(
      OUT / f"{stem}_data.csv", index=False
  )

In [ ]:
def plot_three_panel(df, metric="f1", raid_metric="recall_at_5_fpr"):
  frames = coling_raid_frames(df)
  fig = plt.figure(figsize=(2.7 * 4, 2.75))
  gs_left = gridspec.GridSpec(
      1, 3, figure=fig, left=0.0, right=0.725, wspace=0.13
  )
  gs_right = gridspec.GridSpec(1, 1, figure=fig, left=0.775, right=1.0)
  axes = [fig.add_subplot(gs_left[i]) for i in range(3)] + [
      fig.add_subplot(gs_right[0])
  ]
  axes[0].set_facecolor("#f0f0f0")
  xticks = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
  for i, (ax, (key, d)) in enumerate(zip(axes, frames)):
    m = raid_metric if key == "raid" else metric
    draw_panel(ax, d, m, PALETTE_FIG1, TITLE_TEXT[key], 14, 11, i == 0)
    if key != "raid":
      ax.set(xlim=(0.0, 1.0), ylim=(0.5, 1.0), xticks=xticks)
      ax.sharey(axes[0])
      if i != 0:
        ax.tick_params(labelleft=False)
    else:
      ax.set(xlim=(0.0, 1.0), ylim=(0.0, 1.0), xticks=xticks)
    ax.tick_params(labelsize=12.5)
  plt.tight_layout(pad=0)
  plt.subplots_adjust(wspace=0.2)
  return fig, pd.concat([d for _, d in frames])


def plot_four_panel(df, metric="recall_at_1_fpr", palette="viridis"):
  frames = coling_raid_frames(df)
  fig, axes = plt.subplots(1, 4, figsize=(3 * 4, 2.25), sharey=True)
  axes[0].set_facecolor("#f0f0f0")
  for i, (ax, (key, d)) in enumerate(zip(axes, frames)):
    draw_panel(ax, d, metric, palette, TITLE_TEXT[key], 12, 10, i == 0)
    ax.set(
        xlim=(0.0, 1.0), ylim=(0.0, 1.0), yticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    )
  plt.tight_layout(pad=0)
  plt.subplots_adjust(wspace=0.1)
  return fig, pd.concat([d for _, d in frames])


def plot_prdb(df, metric="f1"):
  d = df[df["dataset"] == "PRDB"]
  frames = [
      (s, c, d[(d["split"] == s) & (d["target"] == c)])
      for s in ("gpt4o", "llama")
      for c in [s, "qwen", "claude", "gemini"]
  ]
  fig, axes = plt.subplots(2, 4, figsize=(2.6 * 4, 2.25 * 2), sharey=True)
  for i, (ax, (_, c, dd)) in enumerate(zip(axes.flatten(), frames)):
    title = TITLE_TEXT[c]
    if i % 4 == 0:
      ax.set_facecolor("#f0f0f0")
      title = f"{title} (train)"
    draw_panel(ax, dd, metric, PALETTE_FIG2, title, 13.5, 10.5, i == 0)
    ax.set_yscale("function", functions=SQRT_SCALE)
    ax.set(
        xlim=(0.0, 1.0),
        ylim=(0.4, 1.0),
        yticks=[0.4, 0.6, 0.7, 0.8, 0.9, 1.0],
        xticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    )
    ax.tick_params(labelsize=12.5)
  plt.tight_layout(pad=0)
  plt.subplots_adjust(wspace=0.15, hspace=0.4)
  return fig, pd.concat([dd for _, _, dd in frames])


def plot_raid(df, metric="recall_at_5_fpr", tiling=(3, 4)):
  d = df[df["dataset"] == "RAID"]
  targets = d["target"].unique().tolist()
  n_rows, n_cols = tiling
  assert n_rows * n_cols == len(targets), targets
  fig, axes = plt.subplots(
      n_rows, n_cols, figsize=(3 * n_cols, 2.75 * n_rows), sharey=True
  )
  for i, (ax, c) in enumerate(zip(axes.flatten(), targets)):
    draw_panel(ax, d[d["target"] == c], metric, PALETTE_FIG1, c, 12, 10, i == 0)
    ax.set(xlim=(0.0, 1.0), ylim=(0.0, 1.0))
    ax.set_yscale("function", functions=SQRT_SCALE)
  plt.tight_layout(pad=0)
  plt.subplots_adjust(wspace=0.1, hspace=0.4, top=0.85)
  return fig, d

In [ ]:
paper.style()
for head in ("xgboost", "logreg"):
  d = results[
      results["head"].isin(HEAD_FILTER[head])
      & results["base"].isin(["gemma2", "gram2vec"])
  ]
  save(*plot_three_panel(d), f"three_panel{SUFFIX[head]}")
  save(*plot_four_panel(d), f"four_panel{SUFFIX[head]}")
  save(*plot_prdb(d), f"prdb{SUFFIX[head]}")
  save(*plot_raid(d), RAID_STEM[head])

In [ ]:
specs = render.hc_row_specs()
frames = {
    t: stats.head_comparison_cells(
        t, [(f"PRDB_{t}", g) for g in paper.GENERATOR_LABEL], specs
    )
    for t in ("gpt4o", "llama")
}
coling = stats.head_comparison_cells(
    "coling", [(s, "all") for s, _ in render.HC_COLING_SPLITS], specs
)
pd.concat([*frames.values(), coling], ignore_index=True).to_csv(
    TABLES / "head_comparison_cells.csv", index=False
)
render.head_comparison_prdb(frames, TABLES / "head_comparison_prdb.tex")
print(
    render.head_comparison_coling(coling, TABLES / "head_comparison_coling.tex")
)

In [ ]:
rows, summary = stats.dataset_composition()
rows.to_csv(OUT / "dataset_composition_rows.csv", index=False)
summary.to_csv(OUT / "dataset_composition_summary.csv", index=False)
print(render.dataset_composition(summary, OUT / "dataset_composition.tex"))

## Sweep

In [ ]:
sweep = stats.sweep_metrics(paper.SWEEP_ROOT)
gram2vec = stats.sweep_metrics(paper.GRAM2VEC_ROOT)
chosen = pd.concat(
    [
        stats.select_operating_point(sweep, head)
        for head in sorted(sweep["head"].unique())
    ]
    + [stats.select_operating_point(gram2vec, "logreg_gram2vec")]
)
chosen[
    [
        "head",
        "target",
        "split",
        "block",
        "lr",
        "l1",
        "tau",
        "epoch",
        "auroc",
        "f1",
        "nonzero_coefs",
        "active_per_doc",
    ]
]

In [ ]:
FONT = dict(tick=11, label=13, title=13, row_title=12, legend=10)
LR_ORDER = ["1e-03", "3e-03", "1e-02", "3e-02"]
LR_SHORT = {
    "1e-03": r"$10^{-3}$",
    "3e-03": r"$3\!\times\!10^{-3}$",
    "1e-02": r"$10^{-2}$",
    "3e-02": r"$3\!\times\!10^{-2}$",
}
PALETTE_LR = {
    "1e-03": "#2a78d6",
    "3e-03": "#eb6834",
    "1e-02": "#4a3aa7",
    "3e-02": "#e34948",
}
LAMBDA_ORDER = ["1e-02", "3e-02", "1e-01", "3e-01", "1e+00"]
HEAD_LABEL = {
    paper.eval_dir(sae, "logreg"): paper.SAE_LABEL[sae]
    for sae in paper.SAE_ORDER
}
SETTING_ORDER = [paper.SAE_LABEL[sae] for sae in paper.SAE_ORDER]
MASK_ORDER = [paper.TAU_LABEL[0.0], paper.TAU_LABEL[0.7]]

grid = stats.l1_grid(sweep)
grid.to_csv(OUT / "l1_grid_search_f1_data.csv", index=False)
df = (
    grid.assign(
        lambda_f=grid["lambda"].map(float),
        setting=pd.Categorical(
            grid["head"].map(HEAD_LABEL), SETTING_ORDER, ordered=True
        ),
        mask=pd.Categorical(
            grid["tau"].round(2).map(paper.TAU_LABEL), MASK_ORDER, ordered=True
        ),
        lr=pd.Categorical(grid["lr"], LR_ORDER, ordered=True),
        **{
            "lambda": pd.Categorical(grid["lambda"], LAMBDA_ORDER, ordered=True)
        },
    )
    .sort_values(["setting", "mask", "lr", "lambda_f"])
    .reset_index(drop=True)
)
assert (
    len(df) == 120
    and (df.groupby(["setting", "mask"], observed=True).size() == 20).all()
)
for key, g in df.groupby(["setting", "mask", "lr"], observed=True):
  assert g.sort_values("lambda_f")[
      "active_per_doc"
  ].is_monotonic_decreasing, key
render.l1_selected_table(df)

In [ ]:
paper.style()
plt.rcParams.update(
    {
        "font.size": FONT["label"],
        "axes.labelsize": FONT["label"],
        "axes.titlesize": FONT["title"],
        "xtick.labelsize": FONT["tick"],
        "ytick.labelsize": FONT["tick"],
        "legend.fontsize": FONT["legend"],
    }
)
g = sns.relplot(
    data=df,
    kind="line",
    x="active_per_doc",
    y="f1",
    hue="lr",
    hue_order=LR_ORDER,
    palette=PALETTE_LR,
    col="setting",
    col_order=SETTING_ORDER,
    row="mask",
    row_order=MASK_ORDER,
    marker="o",
    markersize=5,
    linewidth=1.6,
    errorbar=None,
    legend=False,
    height=1.7,
    aspect=1.7,
    facet_kws=dict(sharey=False, sharex="col", margin_titles=True),
)
g.set(xscale="log")
g.set_axis_labels("", "")
g.set_titles(
    col_template="{col_name}", row_template="{row_name}", size=FONT["title"]
)
row_titles = list(getattr(g, "_margin_titles_texts", []))
for t in row_titles:
  t.set_size(FONT["row_title"])
  t.set_in_layout(False)


def xlim_for(settings, xpad=1.8, xfloor=1.15):
  d = df[df["setting"].isin(settings)]["active_per_doc"]
  return max(d.min() / xpad, xfloor), d.max() * xpad


rest = SETTING_ORDER[:-1]
xlims = {SETTING_ORDER[-1]: xlim_for([SETTING_ORDER[-1]])} | {
    s: xlim_for(rest) for s in rest
}
for (_, setting), ax in g.axes_dict.items():
  ax.grid(True, which="major", axis="y", lw=0.4, alpha=0.5)
  ax.set_xlim(*xlims[setting])
  ax.xaxis.set_major_locator(LogLocator(base=10.0, numticks=12))
  ax.xaxis.set_minor_locator(
      LogLocator(base=10.0, subs=tuple(range(2, 10)), numticks=100)
  )
  ax.xaxis.set_minor_formatter(NullFormatter())
  ax.margins(y=0.10)
  ax.autoscale(enable=True, axis="y")
  ax.tick_params(axis="y", labelleft=True)
  ax.yaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 5, 10]))
  ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
handles = [
    plt.Line2D(
        [],
        [],
        color=PALETTE_LR[lr],
        marker="o",
        markersize=5,
        lw=1.6,
        label=LR_SHORT[lr],
    )
    for lr in LR_ORDER
]
g.axes[0, 0].legend(
    handles=handles,
    loc="lower right",
    ncol=2,
    title="Learning rate",
    title_fontsize=FONT["legend"],
    fontsize=FONT["legend"],
    frameon=True,
    handlelength=0.9,
    handletextpad=0.3,
    columnspacing=0.6,
    labelspacing=0.15,
    borderpad=0.25,
    borderaxespad=0.15,
)
sns.despine(fig=g.figure)
g.figure.tight_layout(pad=0, w_pad=1.0, h_pad=0.15)
save_kw = dict(bbox_inches="tight", pad_inches=0, bbox_extra_artists=row_titles)
g.savefig(OUT / "l1_grid_search_f1.pdf", **save_kw)
g.savefig(OUT / "l1_grid_search_f1.png", dpi=200, **save_kw)
print(render.l1_params_table(TABLES / "l1_grid_params.tex"))

## Features

In [ ]:
TOP = OUT / f"top_features_tau{paper.TOP_TAU}"
TOP.mkdir(parents=True, exist_ok=True)
for slug in paper.TOP_FEATURE_ORDER:
  render.top_features_dump(
      stats.feature_block(paper.SETTINGS[slug]), paper.SETTINGS[slug], TOP
  )
dumps = {
    slug: pd.read_csv(render.dump_path(TOP, paper.SETTINGS[slug]))
    for slug in paper.TOP_FEATURE_ORDER
}
log = []
render.split_tables(dumps, TOP, log)
print("\n".join(log))

In [ ]:
heads = {}
for slug in ("prdb4o_ft_model_sae", "prdb_ft_model_sae"):
  s = paper.SETTINGS[slug]
  heads[s.target] = stats.lor_join(stats.feature_block(s), s)
print(render.top10_tables(heads, TOP))

In [ ]:
counts = stats.active_feature_counts()
counts.to_csv(TABLES / "active_feature_counts.csv", index=False)
for split in ("train", "test", "realtest"):
  per_doc = {}
  for slug in render.COUNT_SETTINGS:
    s = paper.SETTINGS[slug]
    tag = "test" if split == "realtest" and s.dataset != "coling" else split
    per_doc[slug] = json.loads(
        (
            paper.ACTIVE_PER_DOC / f"{s.train_dir}_{s.target}_{tag}.json"
        ).read_text()
    )
  table = render.active_feature_counts(
      counts, per_doc, TABLES / "active_feature_counts.tex", split
  )
print(table)

In [ ]:
features = stats.described_features()
HIST = OUT / "feature_histograms"
HIST.mkdir(parents=True, exist_ok=True)
cols = [
    "sae_type",
    "dataset",
    "target",
    "layer",
    "old_feature_index",
    "feature_index",
    "correlation",
]
cols += [c for c in features.columns if c.endswith("_lor")]
features[cols].sort_values(cols, kind="stable").to_csv(
    HIST / "features.csv", index=False
)
median_corr = features.pivot_table(
    values="correlation", index="target", columns="sae_type", aggfunc="median"
).round(3)
train_lor = features.apply(
    lambda r: r[f"{stats.in_dist_key(r['target'])}_lor"], axis=1
)
median_lor = (
    features.assign(train_lor=pd.to_numeric(train_lor, errors="coerce").abs())
    .pivot_table(
        values="train_lor", index="target", columns="sae_type", aggfunc="median"
    )
    .round(3)
)
median_corr.to_csv(HIST / "median_correlation.csv")
median_lor.to_csv(HIST / "median_train_abs_lor.csv")
print(median_corr, median_lor, sep="\n\n")

In [ ]:
def set_hist_fonts():
  plt.rcParams["font.sans-serif"] = ["Google Sans", "Arial"]
  plt.rcParams["font.family"] = "sans-serif"


def style_corr_axis(ax, title, xlim=(0.0, 1.0), tick_fontsize=12):
  ax.set_xlim(*xlim)
  ax.set_title(title, fontweight="bold")
  ax.set_xlabel("")
  ax.set_ylabel("")
  ax.tick_params(axis="x", which="both", bottom=True, labelsize=tick_fontsize)
  ax.tick_params(axis="y", which="both", left=False)
  ax.set_yticklabels([])
  ax.grid(axis="y", linestyle="--", alpha=0.7)


sns.set_theme(style="white", palette="colorblind")
set_hist_fonts()
targets = [("coling", "Open Domain"), ("prdb", "Closed Domain")]
fig, axes = plt.subplots(
    len(targets), 1, figsize=(4, 1.75 * len(targets)), sharex=True
)
for i, (ax, (target, title)) in enumerate(zip(axes, targets)):
  subset = features[features["dataset"].str.contains(target)].reset_index(
      drop=True
  )
  sns.histplot(
      data=subset,
      x="correlation",
      hue="sae_type",
      bins=100,
      palette="tab10",
      alpha=0.4,
      stat="density",
      common_norm=False,
      kde=True,
      ax=ax,
  )
  style_corr_axis(ax, title)
  legend = ax.get_legend()
  handles = legend.legend_handles
  labels = [
      paper.SAE_LABEL.get(t.get_text(), t.get_text())
      for t in legend.get_texts()
  ]
  if i == len(targets) - 1:
    legend.remove()
  else:
    ax.legend(
        handles,
        labels,
        title="",
        shadow=True,
        frameon=True,
        framealpha=1.0,
        facecolor="white",
    )
  sns.despine(ax=ax)
plt.tight_layout(pad=0)
plt.subplots_adjust(hspace=0.33)
fig.savefig(OUT / "correlation_hist.pdf", **SAVE_KW)
plt.close(fig)

In [ ]:
LOR_SAE_ORDER = [("ft_model_sae", 1), ("ft_sae", 0), ("base_sae", 2)]
LOR_TITLES = {
    "coling": "Open Domain",
    "gpt4o": "Closed Domain (GPT4o)",
    "llama": "Closed Domain (Llama)",
}


def lor_panel(ax, subset, key, title, bins, alpha, legend):
  palette = sns.color_palette("tab10", n_colors=3)
  for sae_type, color_idx in LOR_SAE_ORDER:
    rows = subset[subset["sae_type"] == sae_type]
    if not rows.empty:
      lors = pd.to_numeric(rows[f"{key}_lor"], errors="coerce").abs()
      ax.hist(
          lors,
          color=palette[color_idx],
          label=paper.SAE_LABEL[sae_type],
          alpha=alpha,
          bins=bins,
          linewidth=0.0,
      )
  ax.set_title(title, fontweight="semibold", fontsize=15)
  ax.set_yscale("log")
  ax.set_xlim(left=0)
  ax.tick_params(axis="x", which="both", bottom=True, labelsize=14)
  ax.tick_params(axis="y", which="both", left=True, labelsize=14)
  ax.tick_params(axis="both", labelsize=13)
  if legend:
    ax.legend(fontsize=10, shadow=True, frameon=True, title=None)


def lor_figure(panels, shape, alpha, path, bins=10):
  set_hist_fonts()
  fig, axes = plt.subplots(*shape, figsize=(9, 2 * shape[0]), sharey=True)
  for i, (ax, (subset, key, title)) in enumerate(zip(axes.flatten(), panels)):
    lor_panel(ax, subset, key, title, bins, alpha, i == 0)
  plt.tight_layout(pad=0)
  plt.subplots_adjust(wspace=0.08)
  fig.savefig(path, **SAVE_KW)
  plt.close(fig)


targets = list(dict.fromkeys(features["target"]))
lor_figure(
    [
        (features[features["target"] == t], stats.in_dist_key(t), LOR_TITLES[t])
        for t in targets
    ],
    (1, len(targets)),
    0.6,
    OUT / "lor_hist.pdf",
)
coling_keys = [p for p, _, _ in stats.LOR_SPLITS["coling"]]
coling_features = features[features["dataset"] == "coling"]
lor_figure(
    [(coling_features, k, k) for k in coling_keys],
    (1, len(coling_keys)),
    0.6,
    OUT / "lor_ood.pdf",
)
prdb_keys = [p for p, _, _ in stats.LOR_SPLITS["prdb"]]
lor_figure(
    [
        (features[features["dataset"] == d], k, k)
        for d in ("prdb_gpt4o", "prdb_llama")
        for k in prdb_keys
    ],
    (2, len(prdb_keys)),
    0.4,
    OUT / "lor_ood_prdb.pdf",
)

## Worked examples

In [ ]:
WE = OUT / "worked_examples"
INPUTS = paper.WORKED_EXAMPLES / "paraphrase_inputs"
# (analyze json, directory, prefix, top_k, setting, max tokens, document source, documents)
RECIPES = [
    (
        "t240rest_ftsae_tau0.85.json",
        "t240_ai4_ftsae085",
        "df",
        7,
        "prdb4o_ft_sae",
        240,
        "gpt4o_rest_t240.jsonl",
        "gpt-4o-batch-ICLR2019-826-Syl58eh927,gpt-4o-batch-NeurIPS2019-668-a2841f0fab,gpt-4o-batch-ICLR2019-429-Byl6BILU27,gpt-4o-batch-ICLR2019-1259-r1g5pn1527,gpt-4o-batch-NeurIPS2016-248-c6d7bae920",
    ),
    (
        "t240_llama_tau0.85.json",
        "t240_llama_tau0.85",
        "llama085",
        7,
        "prdb_ft_model_sae",
        240,
        None,
        "llama-3.1-70b-ICLR2020-733-SklIrvyAKB,llama-3.1-70b-ICLR2017-307-c2997525c6,llama-3.1-70b-ICLR2019-841-BygCDbac37,llama-3.1-70b-ICLR2021-1934-6POeD7QDvXS,llama-3.1-70b-NeurIPS2019-72-8deee42016,llama-3.1-70b-NeurIPS2016-77-ff97f5c48a,human-NeurIPS2016-516-0ec49e3369",
    ),
    (
        "t240_coling2_nonessay_ftsae.json",
        "t240_coling_rand",
        "r_ftsae_nonessay",
        7,
        "coling_ft_sae",
        240,
        "coling_nonessay_ai_t240.jsonl",
        "coling63308_Mixset,coling23806_CUDRT,coling48911_CUDRT,coling17261_CUDRT,coling27468_CUDRT,coling29322_CUDRT,coling38177_NLPeer-F1000",
    ),
    (
        "t240_ftsae_tau0.85.json",
        "t240_ftsae_tau0.85",
        "ftsae085",
        7,
        "prdb4o_ft_sae",
        240,
        None,
        "human-NeurIPS2019-16-af0539fdca,human-ICLR2018-499-3d172f787f,gpt-4o-batch-NeurIPS2016-446-eaa41a15f1,gpt-4o-batch-NeurIPS2016-16-fa366984ee,gpt-4o-batch-NeurIPS2017-362-6bc8a61fb7,gpt-4o-batch-ICLR2017-363-0af4fec13c",
    ),
    (
        "t240_coling_basesae_tau0.85.json",
        "t240_coling_basesae",
        "cs",
        7,
        "coling_base_sae",
        240,
        "coling_sample_t240.jsonl",
        "coling12191_ieltsduck,coling9736_ieltsduck,coling14676_ieltsduck,coling38136_ieltsduck,coling2020_ieltsduck,coling61771_NLPeer-F1000,coling53837_CUDRT,coling53434_CUDRT,coling56295_CUDRT,coling2221_CUDRT",
    ),
    (
        "t240_coling_ftmodel_tau0.85.json",
        "t240_coling_ftmodel",
        "cm",
        7,
        "coling_ft_model_sae",
        240,
        "coling_sample_t240.jsonl",
        "coling39961_CUDRT,coling6597_CUDRT,coling18639_CUDRT,coling40009_peersum,coling11475_CUDRT,coling50538_NLPeer-F1000,coling2020_ieltsduck,coling22518_peersum,coling29355_NLPeer-F1000,coling3478_CUDRT,coling72604_NLPeer-F1000,coling5045_peersum,coling24047_NLPeer-ARR22,coling12300_peersum",
    ),
    (
        "big_ftsae_gpt4o_t140.json",
        "t240_rkxbmCrTKS",
        "rkx",
        8,
        "prdb4o_ft_sae",
        140,
        None,
        "human-ICLR2020-205-rkxbmCrTKS",
    ),
]
for stored, folder, prefix, top_k, slug, max_tokens, source, ids in RECIPES:
  doc_ids = ids.split(",")
  data = stats.worked_example_analysis(
      WE / folder / "analysis.jsonl",
      paper.WORKED_EXAMPLES / stored,
      paper.SETTINGS[slug],
      float(paper.TOP_TAU),
      max_tokens,
      doc_ids,
      INPUTS / source if source else None,
  )
  for path in render.worked_examples(data, WE / folder, prefix, doc_ids, top_k):
    print(path)

In [ ]:
PAIRS = [
    ("coling_base_gemma2_tau0.85_ailm", "doc5217min_fid20122"),
    ("coling_gemma2_tau0.7_round2", "doc133_fid48714"),
    ("coling_gemmascope_tau0.7_round2", "doc211_fid22735"),
    ("prdb_gemmascope_gpt4o_tau0.7_spec", "doc158_fid13630"),
    ("prdb_gemma2_gpt4o_tau0.7_spec", "doc423_fid25706"),
    ("prdb_gemma2_gpt4o_tau0.85_round5", "doc4691A_fid32334"),
    ("prdb_gemma2_gpt4o_tau0.85_round6", "doc45579v4_fid32334"),
]
FAVORITES = WE / "favorites"
render.write(
    FAVORITES / "paraphrase_example_macros.tex", render.PARAPHRASE_MACROS
)
for spec, pair_id in PAIRS:
  data = json.loads(
      (paper.WORKED_EXAMPLES / f"paraphrase_{spec}.json").read_text()
  )
  inputs = stats.read_jsonl(INPUTS / f"{spec}.jsonl")
  (pair,) = [
      p
      for p in render.paraphrase_pairs(data, inputs)
      if p["pair_id"] == pair_id
  ]
  render.paraphrase_example(
      pair, data["meta"], path=FAVORITES / f"paraphrase_example_{pair_id}.tex"
  )

## Counterfactual

In [ ]:
stats.build_cache()
CF = OUT / "counterfactual"
CF.mkdir(parents=True, exist_ok=True)
for name, frame in stats.counterfactual_analysis().items():
  frame.to_csv(CF / f"{name}.csv", index=False)
for key in ("prdb", "coling"):
  render.cf_most_frequent(
      pd.read_csv(CF / f"most_frequent_{key}.csv"),
      OUT / "cf_most_frequent" / f"combined_{key}_tau0.70_top6.tex",
  )
pd.read_csv(CF / "feature_agreement_table.csv")

In [ ]:
CF_FONT = dict(tick=9, label=11, title=11, row_title=10, annot=9)
POINT_COLOR, FIT_COLOR, ZERO_COLOR = "#2a78d6", "0.35", "0.6"
ANNOT_HEADROOM, ANNOT_BAND = 0.24, 0.2
COL_ORDER = [paper.SAE_LABEL[s] for s in paper.SAE_ORDER]
ROW_ORDER = [
    f"{ds}\n{paper.TAU_LABEL[t]}"
    for ds in ("COLING", "PRDB")
    for t in paper.TAU_LABEL
]


def padded(lo, hi, frac=0.06, top=None):
  lo, hi = min(0.0, lo), max(0.0, hi)
  span = hi - lo
  return lo - span * frac, hi + span * (frac if top is None else top)


def panel_extras(data, stats_by_stem, color=None, label=None, **kws):
  ax = plt.gca()
  s = stats_by_stem[data["config_stem"].iloc[0]]
  x0, x1 = padded(data["mean_cf"].min(), data["mean_cf"].max())
  y0, y1 = padded(
      data["mean_para"].min(), data["mean_para"].max(), top=ANNOT_HEADROOM
  )
  ax.set_xlim(x0, x1)
  ax.set_ylim(y0, y1)
  ax.axhline(0, color=ZERO_COLOR, lw=0.6, zorder=1)
  ax.axvline(0, color=ZERO_COLOR, lw=0.6, zorder=1)
  y_cap = y1 - ANNOT_BAND * (y1 - y0)
  xs = np.linspace(x0, x1, 400)
  ys = s["slope"] * xs + s["intercept"]
  inside = (ys >= y0) & (ys <= y_cap)
  ax.plot(xs[inside], ys[inside], color=FIT_COLOR, lw=1.0, ls="--", zorder=2)
  ax.text(
      0.97,
      0.96,
      f"$\\rho$ = {s['rho']:.2f}",
      transform=ax.transAxes,
      va="top",
      ha="right",
      fontsize=CF_FONT["annot"],
      bbox=dict(fc="white", ec="none", alpha=0.75, pad=1.5),
      zorder=6,
  )
  ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
  ax.yaxis.set_major_locator(MaxNLocator(nbins=4))


paper.style()
plt.rcParams.update(
    {
        "font.size": CF_FONT["label"],
        "axes.labelsize": CF_FONT["label"],
        "axes.titlesize": CF_FONT["title"],
        "xtick.labelsize": CF_FONT["tick"],
        "ytick.labelsize": CF_FONT["tick"],
    }
)
points = pd.read_csv(CF / "feature_agreement_points.csv")
arm_stats = pd.read_csv(CF / "feature_agreement_stats.csv")
kept = points[points["kept"]].copy()
kept["row"] = kept["dataset"] + "\n" + kept["threshold"].map(paper.TAU_LABEL)
kept["col"] = kept["sae_type"].map(paper.SAE_LABEL)
g = sns.FacetGrid(
    kept,
    row="row",
    col="col",
    row_order=ROW_ORDER,
    col_order=COL_ORDER,
    sharex=False,
    sharey=False,
    height=1.5,
    aspect=1.5,
    margin_titles=True,
    despine=True,
)
g.map_dataframe(
    sns.scatterplot,
    x="mean_cf",
    y="mean_para",
    color=POINT_COLOR,
    alpha=0.65,
    s=16,
    edgecolor="white",
    linewidth=0.4,
    zorder=3,
)
g.map_dataframe(
    panel_extras,
    stats_by_stem=arm_stats.set_index("config_stem").to_dict("index"),
)
g.set_axis_labels("", "")
g.set_titles(
    col_template="{col_name}", row_template="{row_name}", size=CF_FONT["title"]
)
row_titles = list(getattr(g, "_margin_titles_texts", []))
for t in row_titles:
  t.set_size(CF_FONT["row_title"])
  t.set_in_layout(False)
g.figure.tight_layout(pad=0)
save_kw = dict(bbox_inches="tight", pad_inches=0, bbox_extra_artists=row_titles)
g.savefig(OUT / "cf_feature_agreement.pdf", **save_kw)
g.savefig(OUT / "cf_feature_agreement.png", dpi=200, **save_kw)

## Adversarial

In [ ]:
def adversarial_figure(df, settings, path_stem):
  sns.set_theme(
      style="ticks",
      font="Google Sans",
      rc={"font.sans-serif": ["Google Sans", "Arial", "DejaVu Sans"]},
  )
  df = df[df["cond"].isin(paper.FIG_CONDS)].copy()
  df["clump"] = [
      f"{paper.DATASET_LABEL[d]}\n{g}"
      for d, g in zip(df["dataset"], df["group"])
  ]
  df["cond_label"] = df["cond"].map(paper.FIG_COND_LABELS)
  df["flip_pct"] = 100 * df["flip"]
  clumps = [
      (
          paper.SETTINGS[s].dataset,
          f"{paper.DATASET_LABEL[paper.SETTINGS[s].dataset]}\n{paper.SAE_LABEL[paper.SETTINGS[s].sae]}",
      )
      for s in settings
  ]
  panels = list(dict.fromkeys(ds for ds, _ in clumps))
  short = {"coling": "COLING", "prdb": "PRDB", "prdb_gpt4o": "PRDB"}
  fig, axes = plt.subplots(1, len(panels), figsize=(6.2, 1.6), sharey=False)
  axes = np.atleast_1d(axes)
  w = 0.84 / len(paper.FIG_CONDS)
  for ax, ds in zip(axes, panels):
    sub = df[df["dataset"] == ds]
    cl_ds = [c for d, c in clumps if d == ds]
    sns.barplot(
        data=sub,
        x="clump",
        y="flip_pct",
        hue="cond_label",
        order=cl_ds,
        hue_order=[paper.FIG_COND_LABELS[c] for c in paper.FIG_CONDS],
        palette={
            paper.FIG_COND_LABELS[c]: paper.COND_COLORS[c]
            for c in paper.FIG_CONDS
        },
        errorbar=None,
        width=0.84,
        edgecolor="none",
        linewidth=0,
        ax=ax,
        zorder=3,
        legend=(ax is axes[-1]),
    )
    for ci, c in enumerate(paper.FIG_CONDS):
      for xi, cl in enumerate(cl_ds):
        r = sub[(sub["clump"] == cl) & (sub["cond"] == c)]
        if r.empty:
          continue
        r = r.iloc[0]
        ax.errorbar(
            xi - 0.42 + w * (ci + 0.5),
            100 * r.flip,
            yerr=[[100 * (r.flip - r.flip_lo)], [100 * (r.flip_hi - r.flip)]],
            fmt="none",
            ecolor="#333333",
            elinewidth=0.6,
            capsize=1.2,
            zorder=4,
        )
    ymax = min(100, int(np.ceil(100 * float(sub["flip_hi"].max()) / 10)) * 10)
    ax.set_ylim(0, ymax)
    ax.set_yticks(
        [0, 25, 50, 75, 100]
        if ymax == 100
        else [t for t in (0, 20, 40, 60, 80) if t <= ymax]
    )
    ax.set_xticks(range(len(cl_ds)))
    ax.set_xticklabels(
        [
            paper.SAE_LABEL[paper.SETTINGS[s].sae]
            for s in settings
            if paper.SETTINGS[s].dataset == ds
        ],
        fontsize=9,
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=9.5, length=2.5, pad=1.5)
    ax.tick_params(axis="x", length=2.5, pad=1.0)
    ax.yaxis.grid(True, color="#e5e5e5", linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)
    ax.spines["bottom"].set_zorder(5)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.set_title(
        f"% Flipped ({short[ds]})", fontsize=11, fontweight="normal", pad=3
    )
  handles, labels = axes[-1].get_legend_handles_labels()
  axes[-1].get_legend().remove()
  leg = axes[-1].legend(
      handles,
      labels,
      loc="center left",
      bbox_to_anchor=(1.0, 0.5),
      ncol=1,
      fontsize=8.2,
      frameon=False,
      handlelength=1.0,
      handletextpad=0.5,
      labelspacing=0.35,
      borderaxespad=0.15,
      borderpad=0.0,
  )
  for txt in leg.get_texts():
    if txt.get_text().startswith("SAE"):
      txt.set_fontweight("bold")
  fig.tight_layout(pad=0, w_pad=0)
  fig.subplots_adjust(wspace=0.11)
  for ext in ("pdf", "png"):
    fig.savefig(
        f"{path_stem}.{ext}",
        format=ext,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.01,
    )
  plt.close(fig)

In [ ]:
ADV = OUT / "adversarial_paraphrase/full"
ADV.mkdir(parents=True, exist_ok=True)
main = stats.adversarial_figure_rows(paper.MAIN_SETTINGS)
main.to_csv(ADV / "paper_f1_bars_data.csv", index=False)
adversarial_figure(main, paper.MAIN_SETTINGS, ADV / "paper_f1_bars")
render.adversarial_table(
    main,
    ["coling"],
    ADV / "table_appendix_coling.tex",
    "tab:adv_appendix_coling",
)
render.adversarial_all_settings(
    stats.adversarial_figure_rows(paper.SETTING_ORDER),
    ADV / "table_appendix_all.tex",
)
gpt4o = stats.adversarial_figure_rows(paper.GPT4O_SETTINGS)
gpt4o.to_csv(ADV / "paper_f1_bars_gpt4o_data.csv", index=False)
adversarial_figure(gpt4o, paper.GPT4O_SETTINGS, ADV / "paper_f1_bars_gpt4o")
render.adversarial_table(
    gpt4o,
    ["coling", "prdb_gpt4o"],
    ADV / "table_appendix_coling_prdb_gpt4o.tex",
    "tab:adv_appendix_coling_prdb_gpt4o",
)
main.pivot(index="setting", columns="cond", values="f1").reindex(
    paper.MAIN_SETTINGS
)[paper.TABLE_CONDS]

In [ ]:
for suffix, settings in (
    ("", paper.MAIN_SETTINGS),
    ("_gpt4o", paper.GPT4O_SETTINGS),
):
  rows = stats.adversarial_rows(
      settings, paper.COND_ORDER, paper.SUMMARY_METRICS
  )
  print(
      render.adversarial_summary(rows, settings, ADV / f"summary{suffix}.csv")
  )